In [ ]:
import os
import math
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from dataclasses import dataclass

# Reproducibility
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# PEMS-BAY: sampled every 5 minutes
STEP_MINUTES = 5

# Forecast setting
INPUT_LEN = 12   # past 12 * 5min = 60 minutes
HORIZON   = 12   # predict next 12 * 5min = 60 minutes

BATCH_SIZE = 64
EPOCHS = 5        # keep it small for class demo
LR = 1e-3


DEVICE: cpu


In [ ]:
def load_pickle(pickle_file):
    with open(pickle_file, "rb") as f:
        try:
            return pickle.load(f)
        except UnicodeDecodeError:
            f.seek(0)
            return pickle.load(f, encoding="latin1")

DATA_DIR = "/content"  # or "/content/data" if you put files there

H5_PATH  = os.path.join(DATA_DIR, "pems-bay.h5")
ADJ_PATH = os.path.join(DATA_DIR, "adj_mx_bay.pkl")

# Load speed time series: shape (T, N)
df = pd.read_hdf(H5_PATH)
print("df shape:", df.shape)
print("time range:", df.index.min(), "->", df.index.max())

speed = df.values.astype(np.float32)   # (T, N)
timestamps = df.index                  # datetime index

# Load adjacency pkl: (sensor_ids, sensor_id_to_ind, adj_mx)
sensor_ids, sensor_id_to_ind, adj_mx = load_pickle(ADJ_PATH)

print("num sensors:", len(sensor_ids))
print("adj shape:", adj_mx.shape)


df shape: (52116, 325)
time range: 2017-01-01 00:00:00 -> 2017-06-30 23:55:00
num sensors: 325
adj shape: (325, 325)


In [ ]:
class StandardScaler:
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std if std > 1e-6 else 1.0

    def transform(self, data):
        return (data - self.mean) / self.std

    def inverse_transform(self, data):
        return (data * self.std) + self.mean

def make_windows(data_2d, input_len=12, horizon=12):
    """
    data_2d: (T, N)
    returns:
      X: (S, input_len, N, 1)
      Y: (S, horizon,   N, 1)
    """
    T, N = data_2d.shape
    X, Y = [], []
    for t in range(input_len, T - horizon + 1):
        x = data_2d[t-input_len:t, :]           # (input_len, N)
        y = data_2d[t:t+horizon, :]             # (horizon, N)
        X.append(x[..., None])                  # add feature dim
        Y.append(y[..., None])
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

# Split by time (classic traffic split)
T = speed.shape[0]
train_end = int(T * 0.7)
val_end   = int(T * 0.8)

train_raw = speed[:train_end]
val_raw   = speed[train_end:val_end]
test_raw  = speed[val_end:]

scaler = StandardScaler(mean=train_raw.mean(), std=train_raw.std())

train_norm = scaler.transform(train_raw)
val_norm   = scaler.transform(val_raw)
test_norm  = scaler.transform(test_raw)

X_train, Y_train = make_windows(train_norm, INPUT_LEN, HORIZON)
X_val,   Y_val   = make_windows(val_norm,   INPUT_LEN, HORIZON)
X_test,  Y_test  = make_windows(test_norm,  INPUT_LEN, HORIZON)

print("X_train:", X_train.shape, "Y_train:", Y_train.shape)
print("X_val  :", X_val.shape,   "Y_val  :", Y_val.shape)
print("X_test :", X_test.shape,  "Y_test :", Y_test.shape)


X_train: (36458, 12, 325, 1) Y_train: (36458, 12, 325, 1)
X_val  : (5188, 12, 325, 1) Y_val  : (5188, 12, 325, 1)
X_test : (10401, 12, 325, 1) Y_test : (10401, 12, 325, 1)


In [ ]:
def sym_adj(adj):
    """Symmetric normalization: D^{-1/2} A D^{-1/2}"""
    adj = np.array(adj, dtype=np.float32)
    rowsum = adj.sum(axis=1)
    d_inv_sqrt = np.power(rowsum, -0.5, where=rowsum>0)
    d_inv_sqrt[~np.isfinite(d_inv_sqrt)] = 0.0
    D = np.diag(d_inv_sqrt)
    return D @ adj @ D

# We use one "support": normalized adjacency
A = sym_adj(adj_mx)
A_torch = torch.tensor(A, dtype=torch.float32, device=DEVICE)
SUPPORTS = [A_torch]


In [ ]:
def masked_mae(pred, real, null_val=0.0):
    mask = (real != null_val).float()
    mask = mask / (mask.mean() + 1e-6)
    loss = torch.abs(pred - real) * mask
    return loss.mean()

def masked_rmse(pred, real, null_val=0.0):
    mask = (real != null_val).float()
    mask = mask / (mask.mean() + 1e-6)
    loss = (pred - real) ** 2 * mask
    return torch.sqrt(loss.mean() + 1e-6)

def masked_mape(pred, real, null_val=0.0):
    mask = (real != null_val).float()
    mask = mask / (mask.mean() + 1e-6)
    loss = torch.abs((pred - real) / (real + 1e-6)) * mask
    return loss.mean()


In [ ]:
class NConv(nn.Module):
    """Graph mixing: X' = A X (applied per feature/channel/time)."""
    def forward(self, x, A):
        # x: (B, C, N, T)   A: (N, N)
        return torch.einsum("bcnt,nm->bcmt", x, A)

class GCN(nn.Module):
    """
    Multi-hop graph mixing (order K) like the Kaggle GWNet code.
    """
    def __init__(self, c_in, c_out, K=2):
        super().__init__()
        self.nconv = NConv()
        self.K = K
        self.mlp = nn.Conv2d((K+1)*c_in, c_out, kernel_size=(1,1))

    def forward(self, x, supports):
        out = [x]
        for A in supports:
            x1 = self.nconv(x, A)
            out.append(x1)
            xk = x1
            for _ in range(2, self.K+1):
                xk = self.nconv(xk, A)
                out.append(xk)
        h = torch.cat(out, dim=1)
        return self.mlp(h)

class GraphWaveNetLite(nn.Module):
    """
    Minimal, intuitive GWNet-like block:
      - temporal gated conv (tanh * sigmoid)
      - graph mixing (GCN)
      - residual + skip connections
      - final 1x1 conv head outputs horizon steps
    """
    def __init__(self, num_nodes, in_dim=1, residual_channels=32, skip_channels=64,
                 end_channels=128, kernel_size=2, blocks=2, layers=2, dropout=0.2):
        super().__init__()
        self.num_nodes = num_nodes
        self.dropout = dropout

        self.start_conv = nn.Conv2d(in_dim, residual_channels, kernel_size=(1,1))

        self.filter_convs = nn.ModuleList()
        self.gate_convs   = nn.ModuleList()
        self.residual_convs = nn.ModuleList()
        self.skip_convs   = nn.ModuleList()
        self.gconvs       = nn.ModuleList()
        self.bns          = nn.ModuleList()

        # dilations like WaveNet: 1,2,4,8,...
        self.receptive_field = 1
        dilation = 1
        for b in range(blocks):
            for i in range(layers):
                self.filter_convs.append(
                    nn.Conv2d(residual_channels, residual_channels, (1, kernel_size), dilation=(1, dilation))
                )
                self.gate_convs.append(
                    nn.Conv2d(residual_channels, residual_channels, (1, kernel_size), dilation=(1, dilation))
                )
                self.residual_convs.append(nn.Conv2d(residual_channels, residual_channels, (1,1)))
                self.skip_convs.append(nn.Conv2d(residual_channels, skip_channels, (1,1)))

                self.gconvs.append(GCN(residual_channels, residual_channels, K=2))
                self.bns.append(nn.BatchNorm2d(residual_channels))

                self.receptive_field += (kernel_size - 1) * dilation
                dilation *= 2

        self.end_conv_1 = nn.Conv2d(skip_channels, end_channels, kernel_size=(1,1))
        self.end_conv_2 = nn.Conv2d(end_channels, HORIZON, kernel_size=(1,1))  # outputs horizon as "channels"

    def forward(self, x, supports):
        """
        x: (B, T_in, N, 1) from our dataset
        Convert to GWNet layout: (B, C, N, T)
        """
        x = x.permute(0, 3, 2, 1)  # (B, 1, N, T)

        # pad if input shorter than receptive field
        if x.size(-1) < self.receptive_field:
            x = F.pad(x, (self.receptive_field - x.size(-1), 0))

        x = self.start_conv(x)  # (B, residual_channels, N, T)
        skip = 0

        for fconv, gconv, rconv, sconv, gcn, bn in zip(
            self.filter_convs, self.gate_convs, self.residual_convs, self.skip_convs, self.gconvs, self.bns
        ):
            residual = x

            # gated temporal conv
            filter_out = torch.tanh(fconv(x))
            gate_out   = torch.sigmoid(gconv(x))
            x = filter_out * gate_out

            x = F.dropout(x, self.dropout, training=self.training)

            # skip
            s = sconv(x)
            skip = s if isinstance(skip, int) else (skip[..., -s.size(-1):] + s)

            # graph mixing
            x = gcn(x, supports)

            # residual connection (align time dims)
            x = x + residual[..., -x.size(-1):]
            x = bn(x)

        x = F.relu(skip)
        x = F.relu(self.end_conv_1(x))
        x = self.end_conv_2(x)  # (B, HORIZON, N, T_out)

        # We only want the last time position
        x = x[..., -1]          # (B, HORIZON, N)
        return x.unsqueeze(-1)  # (B, HORIZON, N, 1)


In [ ]:
def batch_iter(X, Y, batch_size=64, shuffle=True):
    idx = np.arange(X.shape[0])
    if shuffle:
        np.random.shuffle(idx)
    for i in range(0, len(idx), batch_size):
        j = idx[i:i+batch_size]
        yield X[j], Y[j]

num_nodes = speed.shape[1]
model = GraphWaveNetLite(num_nodes=num_nodes).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

def run_eval(X, Y):
    model.eval()
    maes, rmses, mapes = [], [], []
    with torch.no_grad():
        for xb, yb in batch_iter(X, Y, batch_size=BATCH_SIZE, shuffle=False):
            xb_t = torch.tensor(xb, device=DEVICE)
            yb_t = torch.tensor(yb, device=DEVICE)

            pred = model(xb_t, SUPPORTS)  # normalized space
            # convert back to real speeds
            pred_real = scaler.inverse_transform(pred.cpu().numpy())
            y_real    = scaler.inverse_transform(yb)

            pred_t = torch.tensor(pred_real, device=DEVICE)
            y_t    = torch.tensor(y_real, device=DEVICE)

            maes.append(masked_mae(pred_t, y_t).item())
            rmses.append(masked_rmse(pred_t, y_t).item())
            mapes.append(masked_mape(pred_t, y_t).item())

    return float(np.mean(maes)), float(np.mean(rmses)), float(np.mean(mapes))

best_val = 1e9
best_state = None

for epoch in range(1, EPOCHS+1):
    model.train()
    train_losses = []

    for xb, yb in batch_iter(X_train, Y_train, batch_size=BATCH_SIZE, shuffle=True):
        xb_t = torch.tensor(xb, device=DEVICE)
        yb_t = torch.tensor(yb, device=DEVICE)

        pred = model(xb_t, SUPPORTS)
        loss = masked_mae(pred, yb_t)  # train in normalized space

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        train_losses.append(loss.item())

    val_mae, val_rmse, val_mape = run_eval(X_val, Y_val)
    print(f"Epoch {epoch:02d} | train loss {np.mean(train_losses):.4f} | val MAE {val_mae:.3f} RMSE {val_rmse:.3f} MAPE {val_mape:.3f}")

    if val_mae < best_val:
        best_val = val_mae
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

# load best
model.load_state_dict(best_state)
print("Best val MAE:", best_val)


KeyboardInterrupt: 

In [ ]:
test_mae, test_rmse, test_mape = run_eval(X_test, Y_test)
print(f"TEST | MAE {test_mae:.3f} RMSE {test_rmse:.3f} MAPE {test_mape:.3f}")


In [ ]:
def neighbor_list(A, thr=0.0):
    n = A.shape[0]
    neigh = []
    for i in range(n):
        nbrs = np.where(A[i] > thr)[0]
        nbrs = nbrs[nbrs != i]   # remove self-loop
        neigh.append(nbrs)
    return neigh


In [ ]:
neigh = neighbor_list(adj_mx, thr=0.0)
candidates = [i for i in range(len(neigh)) if len(neigh[i]) >= 2]
start = int(np.random.choice(candidates))
print("start:", start, "degree:", len(neigh[start]))


In [ ]:
def route_simple_path(neigh, start, length=20, seed=0):
    rng = np.random.default_rng(seed)
    route = [start]
    visited = {start}
    cur = start
    for _ in range(length - 1):
        nbrs = [v for v in neigh[cur] if int(v) not in visited]
        if len(nbrs) == 0:
            break
        cur = int(rng.choice(nbrs))
        route.append(cur)
        visited.add(cur)
    return route


In [ ]:
route = route_simple_path(neigh, start, length=20, seed=1)
print("route length:", len(route))
print("route:", route[:20])


In [ ]:
def route_forecast(sample_id, route_nodes):
    """
    Returns a 12-step forecast table for the given route (list of sensor indices).
    Uses mean speed across sensors in the route.
    """
    route_nodes = [int(i) for i in route_nodes]
    model.eval()

    xb = torch.tensor(X_test[sample_id:sample_id+1], device=DEVICE)  # (1, Tin, N, 1)
    yb = Y_test[sample_id:sample_id+1]                               # (1, 12, N, 1)

    with torch.no_grad():
        pred_norm = model(xb, SUPPORTS).cpu().numpy()                # (1, 12, N, 1)

    pred = scaler.inverse_transform(pred_norm)[0, :, route_nodes, 0].mean(axis=1)  # (12,)
    true = scaler.inverse_transform(yb)[0, :, route_nodes, 0].mean(axis=1)         # (12,)
    abs_err = np.abs(pred - true)

    base_time = timestamps[val_end + INPUT_LEN + sample_id]
    times = [base_time + pd.Timedelta(minutes=STEP_MINUTES*k) for k in range(1, len(pred)+1)]

    return pd.DataFrame({
        "time": times,
        "pred_speed": pred,
        "true_speed": true,
        "abs_error": abs_err
    })

In [ ]:
table = route_forecast(sample_id=500, route_nodes=route)
table

In [ ]:
from math import radians, sin, cos, asin, sqrt

META_PATH = "/content/pems-bay-meta.h5"

def haversine_miles(lat1, lon1, lat2, lon2):
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * asin(sqrt(a)) * 3956  # miles

# Load metadata
meta = pd.read_hdf(META_PATH)

# Make IDs comparable (int)
meta.index = meta.index.astype(int)
sensor_ids_int = np.array([int(s) for s in sensor_ids])

# Map route nodes -> sensor IDs
route_sensor_ids = sensor_ids_int[route]

# Get coords for route sensors that exist in meta
meta_sub = meta.loc[meta.index.intersection(route_sensor_ids), ["Latitude", "Longitude"]]
coords = [(meta_sub.loc[sid, "Latitude"], meta_sub.loc[sid, "Longitude"])
          for sid in route_sensor_ids if sid in meta_sub.index]

# Total distance along the route order
total_miles = sum(
    haversine_miles(coords[i][0], coords[i][1], coords[i+1][0], coords[i+1][1])
    for i in range(len(coords) - 1)
)

# Estimated travel time using average predicted route speed (mph)
avg_speed_mph = float(table["pred_speed"].mean())
travel_time_min = (total_miles / avg_speed_mph) * 60 if avg_speed_mph > 0 else np.nan

print(f"total distance: {total_miles:.3f} miles | average speed: {avg_speed_mph:.2f} mph | estimated travel time: {travel_time_min:.2f} min")

In [ ]:
plt.figure(figsize=(10,4))
plt.scatter(table["time"], table["true_speed"], label="Actual")
plt.scatter(table["time"], table["pred_speed"], label="Predicted")
plt.xticks(rotation=30)
plt.title("Route speed forecast (12 discrete steps, 5-min intervals)")
plt.ylabel("Speed")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10,3))
plt.scatter(table["time"], table["abs_error"], label="Absolute error", color='red')
plt.xticks(rotation=30)
plt.title("Route forecast error (12 discrete steps, 5-min intervals)")
plt.ylabel("Abs error")
plt.legend()
plt.tight_layout()
plt.show()


#Additions

In [ ]:
# =========================
# Additions: OSM route plot + viability check for Dijkstra sensor-path
# =========================

import networkx as nx
from shapely.geometry import LineString

# --- Optional: OSM tools (fast bounding-box download) ---
try:
    import osmnx as ox
except Exception as e:
    ox = None
    print("osmnx not available. Install with: pip install osmnx")
    print("OSM plotting/viability will be skipped. Error:", repr(e))


def _ensure_meta_index_int(meta_df: pd.DataFrame) -> pd.DataFrame:
    m = meta_df.copy()
    m.index = m.index.astype(int)
    return m


def get_route_sensor_coords(route_nodes, sensor_ids, meta_df):
    """
    route_nodes: list of sensor indices (0..N-1)
    sensor_ids: list-like of sensor IDs (as in adj_mx pickle)
    meta_df: indexed by sensor ID, must have Latitude/Longitude columns
    returns: (route_sensor_ids, lats, lons) aligned to route order
    """
    meta_df = _ensure_meta_index_int(meta_df)
    sensor_ids_int = np.array([int(s) for s in sensor_ids])

    route_nodes = [int(i) for i in route_nodes]
    route_sensor_ids = sensor_ids_int[route_nodes]

    lats, lons, kept_ids = [], [], []
    for sid in route_sensor_ids:
        if int(sid) in meta_df.index:
            lats.append(float(meta_df.loc[int(sid), "Latitude"]))
            lons.append(float(meta_df.loc[int(sid), "Longitude"]))
            kept_ids.append(int(sid))

    return np.array(kept_ids, dtype=int), np.array(lats, dtype=float), np.array(lons, dtype=float)


def dijkstra_sensor_route(adj_mx, route_sensor_ids, lats, lons, start_k=0, end_k=-1):
    """
    Build a Dijkstra path over the SENSOR GRAPH, but weight edges by haversine distance
    between sensor coordinates (more realistic than 1/adj).
    - Uses only nodes that exist in (route_sensor_ids, lats, lons) arrays.

    Returns: list of sensor-graph indices (0..N-1) representing the path in original graph.
    """
    # Map sensor ID -> original node index in adj matrix
    sensor_id_to_node = {int(sensor_ids[i]): i for i in range(len(sensor_ids))}

    # Nodes we can use (those with coords)
    usable_nodes = [sensor_id_to_node[int(sid)] for sid in route_sensor_ids if int(sid) in sensor_id_to_node]

    if len(usable_nodes) < 2:
        raise ValueError("Not enough route sensors with coords to build Dijkstra path.")

    # Choose endpoints among the usable list
    s_node = usable_nodes[int(start_k)]
    t_node = usable_nodes[int(end_k)]

    # Build a weighted NX graph from adj_mx (sparse add)
    Gs = nx.Graph()
    Gs.add_nodes_from(usable_nodes)

    # Build quick lookup for lat/lon by original node index
    # (only for usable nodes)
    node_to_latlon = {}
    # route_sensor_ids/lats/lons are aligned, but we need mapping by original node index:
    for sid, lat, lon in zip(route_sensor_ids, lats, lons):
        if int(sid) in sensor_id_to_node:
            node_to_latlon[sensor_id_to_node[int(sid)]] = (float(lat), float(lon))

    def edge_meters(u, v):
        (lat1, lon1) = node_to_latlon[u]
        (lat2, lon2) = node_to_latlon[v]
        return haversine_miles(lat1, lon1, lat2, lon2) * 1609.344

    # Add edges where adjacency > 0 AND both endpoints have coords
    # To keep this fast, iterate only over usable nodes.
    adj = adj_mx
    for u in usable_nodes:
        if u not in node_to_latlon:
            continue
        nbrs = np.where(adj[u] > 0)[0]
        for v in nbrs:
            if v == u:
                continue
            if v in node_to_latlon and v in Gs.nodes:
                # undirected
                w = edge_meters(u, v)
                # keep smallest if duplicates
                if Gs.has_edge(u, v):
                    if w < Gs[u][v]["weight"]:
                        Gs[u][v]["weight"] = w
                else:
                    Gs.add_edge(u, v, weight=w)

    if not nx.has_path(Gs, s_node, t_node):
        return None

    path = nx.shortest_path(Gs, source=s_node, target=t_node, weight="weight")
    return [int(x) for x in path]


def build_osm_graph_route_corridor(lats, lons, buffer_m=500):
    """
    Download driving graph only within buffer_m of the route polyline.
    Much smaller than a bbox.
    """
    if ox is None:
        return None

    # OSMnx expects polygon in lat/lon (WGS84). We'll create a LineString in lon/lat.
    line = LineString([(float(lon), float(lat)) for lat, lon in zip(lats, lons)])

    # Project to UTM for a true meter buffer, then back to lat/lon
    gdf = ox.projection.project_gdf(ox.geocoder.geocode_to_gdf(None).iloc[0:0])  # empty template
    # Easier: use OSMnx helpers directly
    # Create a GeoDataFrame from the line
    import geopandas as gpd
    gdf_line = gpd.GeoDataFrame(geometry=[line], crs="EPSG:4326")
    gdf_line_proj = ox.projection.project_gdf(gdf_line)
    poly_proj = gdf_line_proj.buffer(buffer_m).iloc[0]
    poly = ox.projection.project_geometry(poly_proj, crs=gdf_line_proj.crs, to_crs="EPSG:4326")[0]

    G = ox.graph_from_polygon(poly, network_type="drive", simplify=True, retain_all=False, truncate_by_edge=True)
    G = ox.add_edge_lengths(G)
    return G


def snap_points_to_osm_nodes(G_osm, lats, lons):
    """
    Returns the nearest OSM node id for each (lat, lon).
    """
    if ox is None:
        return None
    # osmnx nearest_nodes expects X=lon, Y=lat
    return ox.distance.nearest_nodes(G_osm, X=lons, Y=lats)


def check_route_viable_on_osm(G_osm, lats, lons, max_detour_ratio=3.0, max_road_meters=7000):
    """
    For each consecutive pair of sensors:
    - compute OSM shortest-path road distance (meters) (weight='length')
    - compute straight-line distance (meters)
    - fail if no road path OR road distance too long OR detour ratio too high
    Returns: (is_viable: bool, details: dict)
    """
    if ox is None or G_osm is None:
        return False, {"reason": "osmnx not available or OSM graph not built"}

    osm_nodes = snap_points_to_osm_nodes(G_osm, lats, lons)
    if osm_nodes is None or len(osm_nodes) < 2:
        return False, {"reason": "could not snap sensors to OSM nodes"}

    segments = []
    for i in range(len(osm_nodes) - 1):
        u = int(osm_nodes[i])
        v = int(osm_nodes[i + 1])

        # straight-line meters
        straight_m = haversine_miles(lats[i], lons[i], lats[i+1], lons[i+1]) * 1609.344
        straight_m = max(straight_m, 1.0)  # avoid div by 0

        try:
            road_m = nx.shortest_path_length(G_osm, source=u, target=v, weight="length")
        except nx.NetworkXNoPath:
            return False, {"reason": f"no OSM road path between segment {i}->{i+1}"}
        except Exception as e:
            return False, {"reason": f"OSM routing error on segment {i}->{i+1}: {repr(e)}"}

        detour = float(road_m) / float(straight_m)

        segments.append(
            {"i": i, "road_m": float(road_m), "straight_m": float(straight_m), "detour": detour, "u": u, "v": v}
        )

        if road_m > max_road_meters:
            return False, {"reason": f"segment {i}->{i+1} too long on roads ({road_m:.0f}m)", "segments": segments}

        if detour > max_detour_ratio:
            return False, {"reason": f"segment {i}->{i+1} detour too high ({detour:.2f}x)", "segments": segments}

    return True, {"reason": "ok", "segments": segments}


def plot_route_on_osm(G_osm, lats, lons, title="OSM + sensor route", show=True):
    """
    Plots OSM street graph, overlays sensor points, and draws the OSM shortest paths between them.
    """
    if ox is None or G_osm is None:
        print("Skipping plot: osmnx/OSM graph unavailable.")
        return

    fig, ax = ox.plot_graph(G_osm, show=False, close=False, bgcolor="white", node_size=0, edge_linewidth=0.6)

    # Sensor points
    ax.scatter(lons, lats, s=30, marker="o", zorder=5)
    for i, (lat, lon) in enumerate(zip(lats, lons)):
        ax.text(lon, lat, str(i), fontsize=8, zorder=6)

    # OSM route lines between consecutive sensors
    osm_nodes = snap_points_to_osm_nodes(G_osm, lats, lons)
    for i in range(len(osm_nodes) - 1):
        u = int(osm_nodes[i])
        v = int(osm_nodes[i + 1])
        try:
            path = nx.shortest_path(G_osm, u, v, weight="length")
            xs = [G_osm.nodes[n]["x"] for n in path]  # lon
            ys = [G_osm.nodes[n]["y"] for n in path]  # lat
            ax.plot(xs, ys, linewidth=2.2, zorder=4)
        except Exception:
            # if a segment fails, just skip drawing it
            pass

    ax.set_title(title)
    if show:
        plt.show()

In [ ]:
# 1) Get coords for the sensors in your existing route (or use dijkstra one below)
meta = pd.read_hdf(META_PATH)
route_sensor_ids, route_lats, route_lons = get_route_sensor_coords(route, sensor_ids, meta)

print("Sensors-with-coords in route:", len(route_sensor_ids), "out of", len(route))

# 2) Build a Dijkstra route between endpoints of that route (more “route found by dijkstra”)
#    You can change endpoints by changing start_k/end_k (index within the route list after coord filtering).
dij_path = dijkstra_sensor_route(adj_mx, route_sensor_ids, route_lats, route_lons, start_k=0, end_k=-1)

if dij_path is None:
    print("Dijkstra sensor-route could not be found (disconnected in sensor graph).")
    route_for_map = route  # fallback
else:
    print("Dijkstra sensor-route length:", len(dij_path))
    route_for_map = dij_path

# 3) Recompute coords for the final route we will plot/check
final_sensor_ids, final_lats, final_lons = get_route_sensor_coords(route_for_map, sensor_ids, meta)

# 4) Build small OSM graph around just this path, check viability, plot
if ox is not None and len(final_lats) >= 2:
    G_osm = build_osm_graph_route_corridor(final_lats, final_lons, buffer_m=600)

    ok, info = check_route_viable_on_osm(
        G_osm,
        final_lats,
        final_lons,
        max_detour_ratio=3.0,   # tweak stricter/looser
        max_road_meters=7000    # tweak based on how far adjacent sensors “should” be
    )

    print("Route viable in real life? ->", "Y" if ok else "N")
    if not ok:
        print("Why:", info.get("reason"))

    plot_route_on_osm(G_osm, final_lats, final_lons, title="OSM streets + sensor route (Dijkstra)")

else:
    print("Skipping OSM viability/plot (osmnx missing or too few points).")

